In [2]:
import pandas as pd

df = pd.read_csv("openfoodfacts_10k_india.csv")

allergen_set = set()

# collect allergens_tags
for tags in df["allergens_tags"].dropna():
    for tag in str(tags).split(","):
        tag = tag.replace("en:", "").strip()
        if tag:
            allergen_set.add(tag)

# collect traces_tags
for tags in df["traces_tags"].dropna():
    for tag in str(tags).split(","):
        tag = tag.replace("en:", "").strip()
        if tag:
            allergen_set.add(tag)

print("Unique allergens found:")
print(sorted(allergen_set))


Unique allergens found:
['10013043000622', 'acesulfame-potassium', 'apple', 'banana', 'bn:end-sesame-seeds', 'celery', 'cereals', 'chicken', 'coconut', 'contains-20-30-mg-of-caffeine', 'contains-added-flavour', 'contains-added-natural-flavours', 'contains-artificial-coconut-flavouring-substances', 'contains-nuts', 'contains-oats', 'duplicate-of-original', 'eggs', 'en-milk-en-peanuts', 'fish', 'fr:avoine', 'fr:erdnusse', 'fr:millet', 'fr:schalenfruchte', 'fruit', 'garlic', 'gelatin', 'gluten', 'iodised-salt', 'lemon-juice', 'lupin', 'made-in-a-facility-that-also-processes', 'made-in-a-factory-that-also-uses-wheat-and-salt-alongside-with-malt', 'may-contain-traces-of-other-tree-nut', 'may-contain-traces-of-treenuts', 'may-contains-peanut', 'milk', 'milk-solids', 'mustard', 'mustard-seed', 'none', 'nuts', 'oats-are-processed-in-a-plant-where-gluten-containing-products-are-manufactured', 'onion', 'orange', 'peach', 'peanuts', 'permitted-antioxidants', 'rose', 'saturated-fat', 'sesame-seeds

In [3]:
from collections import defaultdict

allergen_keywords = defaultdict(set)

for _, row in df.iterrows():
    allergens = str(row.get("allergens_tags", "")).split(",")
    ingredients = str(row.get("ingredients_text", "")).lower().split(",")

    for allergen in allergens:
        allergen = allergen.replace("en:", "").strip()
        if allergen:
            for ing in ingredients:
                allergen_keywords[allergen].add(ing.strip())

# Convert to CSV format
rows = []

for allergen, keywords in allergen_keywords.items():
    for keyword in keywords:
        rows.append([allergen, keyword])

output_df = pd.DataFrame(rows, columns=["allergen", "keyword"])
output_df.to_csv("allergen_keywords_india.csv", index=False)

print("Allergen keyword dataset created 🚀")


Allergen keyword dataset created 🚀


In [5]:
import pandas as pd
import re

# Load your uploaded file
df = pd.read_csv("allergen_keywords_india.csv")

# Drop null rows
df = df.dropna()

# Clean text
df["allergen"] = df["allergen"].str.lower().str.strip()
df["keyword"] = df["keyword"].str.lower().str.strip()

# Remove very long noisy ingredient sentences
df = df[df["keyword"].str.len() < 40]

# Remove 'may contain' noise
df = df[~df["keyword"].str.contains("may contain", na=False)]
df = df[~df["keyword"].str.contains("contains", na=False)]

# Remove special characters
df["keyword"] = df["keyword"].apply(lambda x: re.sub(r'[^a-z0-9\s\-]', '', x))

# Keep only single ingredient words (optional strict filter)
df = df[df["keyword"].str.count(" ") <= 2]

# Remove duplicates
df = df.drop_duplicates()

# Save cleaned file
df.to_csv("clean_allergen_keywords_india.csv", index=False)

print("Clean dataset saved as clean_allergen_keywords_india.csv 🚀")


Clean dataset saved as clean_allergen_keywords_india.csv 🚀


In [6]:
grouped = df.groupby("allergen")["keyword"].apply(list).to_dict()

import json
with open("allergen_database_india.json", "w") as f:
    json.dump(grouped, f, indent=4)

print("JSON allergen database created 🚀")


JSON allergen database created 🚀
